# Thảo luận, Phân tích Lỗi và Kết luận

## 1. Tổng hợp các dấu ấn kỹ thuật trên trọn bộ 25.000 bài báo

1. **Độ phủ trọn vẹn 25.000 bài của 5 thành viên (Milestone 1)**: Sau khi chạy bù `--resume` trên Google Colab CPU-only cho thành viên `TRI (24120470)`, kho dữ liệu hợp nhất đạt **`25.000 / 25.000`** bài báo duy nhất (`0` trùng lặp chéo thành viên), **`41.498`** phiên bản (`41.487` phiên bản đã kiểm toán xác minh nguồn; lọc bỏ hình ảnh ngay trên luồng giải nén giảm **`89,30%`** dung lượng xuống còn `22,72 GiB`), **`976.351`** dòng trích dẫn tầng working (`0` mồ côi) và **`456.149`** cạnh có `ref_arxiv_id` trong bảng `references` chuẩn.
2. **Độ chính xác phân tích phân cấp & khử trùng 2 tầng (Milestone 2)**: Trên `38.554` phiên bản có mã nguồn TeX xác minh, parser đạt **`85,66%` `success`** (`33.025` phiên bản) và **`97,34%` dựng thành công cây cấu trúc** (`37.528` phiên bản, bao gồm `4.503` phiên bản `partial`), chỉ còn `2,66%` (`1.026` phiên bản) lỗi cú pháp nặng; toàn bộ `2.764` phiên bản `empty_no_tex` khớp tuyệt đối với bằng chứng kiểm toán M1. Khử trùng 2 tầng xuyên phiên bản cắt giảm **`35,04%`** phần tử văn bản (`23.099.589 → 15.006.000` dòng `elements`, đạt **`-46,95%`** trên nhóm bài đa phiên bản) và **`35,83%`** mục tài liệu tham khảo (`1.834.650 → 1.177.293` dòng `bibtex`, đạt **`-50,41%`** trên nhóm bài đa phiên bản).
3. **Kiểm chuẩn ngoài bộ nhớ & hiệu năng Google Colab CPU-only**: `validate_project` xác nhận **`structural_valid = True`** (`0` lỗi khóa ngoại, `0` chu trình, `0` thiếu phiên bản trên `22.532.299` cạnh `hierarchy`) với RAM trung bình gia quyền chỉ **`240,00 MiB`** (Peak RSS **`845,34 MiB`**, tổng thời gian 5 phiên cold **`18.692,09` giây** = `5h 11p 32s`, trung bình **`0,7477` giây/bài**).

## 2. Giải phẫu định lượng Top 15 thông điệp chẩn đoán (Phân tách Số phiên bản và Số lượt cảnh báo)

Bảng và biểu đồ dưới đây tách bạch rõ **số phiên bản phân biệt bị ảnh hưởng (`distinct_versions`)** với **tổng số lượt cảnh báo ghi nhận (`occurrences`)** trên toàn bộ `41.498` phiên bản, đồng thời ẩn danh hoàn toàn các tên tệp/khóa trích dẫn riêng lẻ:

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

BOOK_PALETTE = {
    "navy": "#0F4C81",
    "teal": "#0D9488",
    "emerald": "#059669",
    "amber": "#D97706",
    "crimson": "#B31B1B",
    "slate": "#64748B",
    "sky": "#0284C7",
    "grid": "#CBD5E1",
}
MEMBER_COLORS = {
    "DAT": "#0F4C81",
    "TUAN": "#0D9488",
    "TAN": "#D97706",
    "TRI": "#059669",
    "TRIET": "#B31B1B",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "figure.figsize": (9.8, 4.3),
    "figure.dpi": 180,
    "savefig.dpi": 180,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 11.0,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#0F4C81",
    "axes.labelsize": 9.5,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.9,
    "ytick.labelsize": 8.9,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": "--",
    "grid.alpha": 0.42,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "—"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "—"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    bar_cols: list[str] | dict[str, tuple[str, float | None]] | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    bar_color: str = "var(--ah-bar-fill)",
    vmax: float | None = None,
):
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    for c in int_set:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").round().astype("Int64")

    fmt_map: dict[str, object] = {}
    for col in out.columns:
        if col in custom_fmt:
            fmt_map[col] = custom_fmt[col]
        elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
            fmt_map[col] = fmt_int
        elif col in pct_set:
            fmt_map[col] = lambda v, d=precision, s=pct_scale100: fmt_pct(v, d, s)
        elif pd.api.types.is_float_dtype(out[col]):
            fmt_map[col] = lambda v, d=precision: fmt_num(v, d)

    styler = out.style.hide(axis="index").set_table_attributes('class="dataframe ah-styled-table"')
    if caption:
        styler = styler.set_caption(caption)
    if fmt_map:
        styler = styler.format(fmt_map)

    if isinstance(bar_cols, dict):
        bar_spec = bar_cols
    else:
        bar_spec = {c: (bar_color, vmax) for c in (bar_cols or [])}

    for bc, (bcol, bmax) in bar_spec.items():
        if bc in out.columns:
            num_s = pd.to_numeric(out[bc], errors="coerce")
            hi = float(bmax) if bmax is not None else float(num_s.max() or 1.0)
            styler = styler.bar(subset=[bc], color=bcol, vmin=0.0, vmax=max(hi, 1e-9))

    num_cols = [
        c for c in out.columns
        if c in int_set or c in pct_set or c in bar_spec or pd.api.types.is_numeric_dtype(out[c])
    ]
    if num_cols:
        styler = styler.set_properties(
            subset=num_cols,
            **{"text-align": "right", "font-variant-numeric": "tabular-nums"},
        )
    return styler


print(
    f"Toàn nhóm (5 thành viên): {fmt_int(scope['papers'])} bài báo / "
    f"{fmt_int(scope['versions'])} phiên bản | Cập nhật: {data['report_date']}"
)

In [ ]:
top_d = pd.DataFrame(m2["top_diagnostics"])
top_d["% / 41.498 version"] = 100.0 * top_d["distinct_versions"] / scope["versions"]
top_view = top_d.rename(columns={
    "message": "Mẫu thông điệp chẩn đoán (đã chuẩn hóa)",
    "distinct_versions": "Số phiên bản phân biệt",
    "occurrences": "Tổng số lượt cảnh báo",
    "primary_status": "Trạng thái chính",
})[["Mẫu thông điệp chẩn đoán (đã chuẩn hóa)", "Số phiên bản phân biệt", "% / 41.498 version", "Tổng số lượt cảnh báo", "Trạng thái chính"]]
display(style_table(
    top_view,
    caption="Bảng 12.1. Top 15 mẫu thông điệp chẩn đoán trên 41.498 phiên bản (phân tách rạch ròi số phiên bản và tổng số lượt cảnh báo).",
    int_cols=["Số phiên bản phân biệt", "Tổng số lượt cảnh báo"],
    pct_cols=["% / 41.498 version"],
    bar_cols={"Số phiên bản phân biệt": ("var(--ah-bar-fill)", None)},
))

top10 = top_d.head(10).iloc[::-1]
short_lbl = [m[:42] + ("…" if len(m) > 42 else "") for m in top10["message"]]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.4, 4.3), sharey=True)
ax1.barh(short_lbl, top10["distinct_versions"], color=BOOK_PALETTE["navy"], height=0.60)
ax1.set_xlim(0, top10["distinct_versions"].max() * 1.25); ax1.grid(axis="x")
ax1.set_xlabel("Số phiên bản phân biệt bị ảnh hưởng")
ax1.set_title("Hình 12.1a. Theo số phiên bản phân biệt")
for idx_v, v_v in enumerate(top10["distinct_versions"]):
    ax1.text(v_v + 18, idx_v, fmt_int(v_v), va="center", fontsize=8.0, fontweight="bold")

ax2.barh(short_lbl, top10["occurrences"], color=BOOK_PALETTE["amber"], height=0.60)
ax2.set_xscale("log"); ax2.set_xlim(10, 25000); ax2.grid(axis="x")
ax2.set_xlabel("Tổng số lượt cảnh báo phát sinh (thang log10)")
ax2.set_title("Hình 12.1b. Theo tổng số lượt xuất hiện")
for idx_o, v_o in enumerate(top10["occurrences"]):
    ax2.text(v_o * 1.15, idx_o, fmt_int(v_o), va="center", fontsize=8.0, fontweight="bold")
plt.tight_layout(); plt.show()

- **`item outside supported list` (`9.087` lượt cảnh báo nhưng chỉ nằm trên `791` phiên bản phân biệt — trạng thái `partial`)**: Phát sinh khi một bài báo lặp lại nhiều lệnh `\item` bên trong môi trường danh sách/thuật toán tự định nghĩa bằng macro ngoài tập chuẩn (`itemize`, `enumerate`, `description`, `compactitem`, `inparaenum`, `list`, `trivlist`). Thay vì loại bỏ cả tài liệu, parser hạ trạng thái của đúng `791` phiên bản này (`1,91%` tổng phiên bản) sang `partial` để giữ trọn phần cây cấu trúc hợp lệ.
- **`Unclosed TeX group` (`285` phiên bản — trạng thái `error`) và `unmatched end environment: <env>` (`331` lượt trên `90` phiên bản phân biệt)**: Bắt nguồn từ ngoặc nhọn không cân bằng hoặc đóng/mở môi trường toán tùy biến từ phía tác giả.
- **Tham chiếu `.bib`/`.bbl`, đa ứng viên root & `\input` động**: Gồm `missing include: <relative_path>` (`3.477` lượt trên `805` phiên bản), `missing/unsafe bibliography: <bib_file>` (`765` lượt trên `637` phiên bản khai báo `.bib` ngoài gói nguồn), `unsupported/empty compiled bibliography: <bbl_file>` (`621` phiên bản), `Expected one compilation root` (`614` phiên bản) và `dynamic/unsupported include in <tex_file>` (`574` lượt trên `271` phiên bản dùng macro trong đường dẫn `\input`).

## 3. Kết luận và cầu nối sang Milestone 3 & Bộ dữ liệu công khai 6 bảng

:::{admonition} Khép kín chu trình sang Milestone 3 & Bộ dữ liệu Hugging Face (6 bảng Parquet — 3,92 GiB)
:class: tip

Từ đầu ra 5 bảng chuẩn hóa của cuốn báo cáo này, nhóm đã hoàn tất trọn vẹn **Milestone 3 — Reference Matching Pipeline** (`src/matching/`) và xuất bản bộ dữ liệu công khai cuối kỳ:
- **Liên thông thực chất từ cây cấu trúc M2 sang khớp trích dẫn M3**: Nhờ khử trùng BibTeX 2 tầng kết hợp đổi tên `\cite{}` xuyên phiên bản, **`98,05%`** mục BibTeX truy vết được trực tiếp câu văn trích dẫn và mục cha trong cây `elements`/`hierarchy` M2. Bộ **25 đặc trưng học máy cổ điển** (`0%` LLM / `0%` pretrained; gồm 4 đặc trưng ngữ cảnh cây Hierarchy M2) huấn luyện trên **`3.022` bài báo gán nhãn** (`25` bài Manual / `1.150` mục + `2.997` bài Auto — `12,0%` / `161.319` mục = **`162.469` dòng `predictions`**) giúp **`LightGBM Ranker`** đạt đúng trần lý thuyết **`MRR@5 = 0,6907`** trên tập Test chuẩn (`373/540`; **`Test-Manual = 0,8885`**, **`Test-Auto = 0,5071`**, `100,0%` Recall@1 trên toàn bộ `373` mục có đích arXiv). Khi kiểm định áp lực khuyết tiêu đề `.bbl` (`S3`), ngữ cảnh cây M2 nâng kết quả lên đỉnh **`0,6802`** trên Test (`+0,0716` so với Heuristic) và **`0,8530 ± 0,0163`** trên 5-Fold Manual CV (`Recall@1 = 96,05%` so với Heuristic `81,44%`).
- **Bộ dữ liệu 6 bảng Parquet phân mảnh trên Hugging Face (`38` shards, `39.359.210` dòng, `3,92 GiB`)**: Xuất bản tại **[nquocdat06/arxiv-harvester-data](https://huggingface.co/datasets/nquocdat06/arxiv-harvester-data)** (`Preview / Viewer / Search / Filter / Statistics = True`). Như đã đối chiếu tại Bảng 8.4, việc gộp `75.000` tệp Parquet nhỏ của M2 (`3,89 GiB` thuần / `3,93 GiB` thư mục `parsed/`) thành `35` shards lớn chuẩn Snappy + Dictionary Encoding giúp giảm `55,33 MiB` overhead, nên trọn bộ **6 bảng** (cộng thêm `papers`, `references` và `predictions`) đạt đúng **`3,92 GiB`** (`4.207.534.396` byte).
- **Mã nguồn & Notebook phân tích thực nghiệm M3**: Xem tại **[NguyenQuocDat06-AI/arxiv-harvester](https://github.com/NguyenQuocDat06-AI/arxiv-harvester)** và **[`docs/09_m3_feature_eda_and_matching.ipynb`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/docs/09_m3_feature_eda_and_matching.ipynb)**.
:::

## Tài liệu và mã nguồn tham chiếu

- [Đặc tả chính thức Milestone 2 — Hierarchical Parsing & Standardization](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) & [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/parser/validation.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/parser/validation.py).
- Mã nguồn triển khai: [`tools/m2_team_consolidator.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m2_team_consolidator.py).
- Mã nguồn triển khai: [`tools/package_hf_dataset.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/package_hf_dataset.py).
- Mã nguồn triển khai: [`docs/09_m3_feature_eda_and_matching.ipynb`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/docs/09_m3_feature_eda_and_matching.ipynb).
- Mã nguồn triển khai: [`tools/build_team_25000_book.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/build_team_25000_book.py).